# Model 1 — 21-brand knowledge distillation on a server

Notebook này điều phối một thí nghiệm có kiểm soát: `YOLOv8s` teacher, `YOLOv8n` baseline và `YOLOv8n` student dùng knowledge distillation (class logits + DFL). Tất cả checkpoint dùng đúng thứ tự 21 class của `data.yaml` nguồn; stable registry ID vẫn được lưu riêng trong `mapping.json`.

Notebook không dùng test split để chọn model. Mọi cờ thực thi mặc định đều là `False`.

## Chuẩn bị trên server

1. Clone repository và checkout phiên bản chứa notebook này.
2. Copy toàn bộ `data/brand_21brands_yolo_v2/` lên đúng vị trí tương đối trong repository.
3. Chọn kernel Python có GPU/CUDA phù hợp.
4. Chạy từng cell theo thứ tự. Không bật nhiều cờ training cùng lúc.

Fingerprint dataset mong đợi: `sha256:8ed180b7ac1def9732f2e61452c6d301a814ef32a62b17517cbd723767d0d492`.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'configs/brand_21brands_yolo_kd.yaml').is_file():
            return candidate
    raise FileNotFoundError(
        'Không tìm thấy repository. Hãy đặt notebook trong repository train_yolo_U.'
    )

PROJECT_ROOT = find_repository_root(Path.cwd().resolve())
os.chdir(PROJECT_ROOT)
print('Repository:', PROJECT_ROOT)
print('Python:', sys.executable)

## Cấu hình và cờ an toàn

Đầu tiên chỉ bật `RUN_INSTALL_DEPS` nếu environment chưa có dependency, sau đó restart kernel. Bật `RUN_PREFLIGHT` trước mọi training. Với mỗi lần chạy chỉ bật đúng một cờ train/resume.

In [ ]:
KD_CONFIG = 'configs/brand_21brands_yolo_kd.yaml'
BASE_CONFIG_OVERRIDE = ''  # Sau pilot tuning, có thể đặt tới recommended_full_config.yaml
HYPERPARAMETER_SEARCH_CONFIG = 'configs/brand_21brands_yolo_search.yaml'
INFERENCE_SEARCH_CONFIG = 'configs/brand_21brands_inference_search.yaml'
EXPECTED_DATASET_FINGERPRINT = 'sha256:8ed180b7ac1def9732f2e61452c6d301a814ef32a62b17517cbd723767d0d492'

TEACHER_CHECKPOINT = 'runs/model1_brand21_distillation/teacher_yolov8s_100ep_s42/weights/best.pt'
BASELINE_CHECKPOINT = 'runs/model1_brand21_distillation/student_yolov8n_baseline_100ep_s42/weights/best.pt'
KD_STUDENT_CHECKPOINT = 'runs/model1_brand21_distillation/student_yolov8n_kd_100ep_s42/weights/best_portable.pt'
TEACHER_LAST = 'runs/model1_brand21_distillation/teacher_yolov8s_100ep_s42/weights/last.pt'
BASELINE_LAST = 'runs/model1_brand21_distillation/student_yolov8n_baseline_100ep_s42/weights/last.pt'
KD_STUDENT_LAST = 'runs/model1_brand21_distillation/student_yolov8n_kd_100ep_s42/weights/last.pt'

RUN_INSTALL_DEPS = False
RUN_AUGMENTATION_PREVIEW = False
RUN_TUNING_PREFLIGHT = False
RUN_HYPERPARAMETER_TUNING = False
RUN_PREFLIGHT = False
RUN_TEACHER = False
RUN_RESUME_TEACHER = False
RUN_BASELINE = False
RUN_RESUME_BASELINE = False
RUN_STUDENT_KD = False
RUN_RESUME_KD = False
RUN_COMPARE = False
RUN_INFERENCE_PREFLIGHT = False
RUN_INFERENCE_CALIBRATION = False

training_flags = [
    RUN_TEACHER, RUN_RESUME_TEACHER, RUN_BASELINE, RUN_RESUME_BASELINE,
    RUN_STUDENT_KD, RUN_RESUME_KD, RUN_HYPERPARAMETER_TUNING,
]
assert sum(bool(value) for value in training_flags) <= 1, 'Chỉ bật một cờ train/resume mỗi lần.'

In [ ]:
install_command = [sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt']
print('Install command:', subprocess.list2cmdline(install_command))
if RUN_INSTALL_DEPS:
    subprocess.run(install_command, cwd=PROJECT_ROOT, check=True)
    print('Đã cài dependency. Hãy restart kernel trước khi tiếp tục.')
else:
    print('Không cài dependency: RUN_INSTALL_DEPS=False')

## Xác minh dataset v2 và class order

Cell này chỉ đọc metadata, không train và không inference. Training ID phải là thứ tự nguồn; `registry_id` là identity ổn định dùng bởi ứng dụng.

In [ ]:
import yaml

data_root = PROJECT_ROOT / 'data/brand_21brands_yolo_v2'
required = [
    data_root / 'data.yaml', data_root / 'mapping.json',
    data_root / 'dataset_checks.json', data_root / 'dataset_fingerprint.json',
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError(f'Dataset v2 chưa đầy đủ: {missing}')
data_yaml = yaml.safe_load((data_root / 'data.yaml').read_text(encoding='utf-8'))
mapping = json.loads((data_root / 'mapping.json').read_text(encoding='utf-8'))
checks = json.loads((data_root / 'dataset_checks.json').read_text(encoding='utf-8'))
names_value = data_yaml['names']
training_names = list(names_value.values()) if isinstance(names_value, dict) else list(names_value)
assert len(training_names) == 21
assert mapping['class_order'] == 'source_data_yaml'
assert checks['dataset_fingerprint'] == EXPECTED_DATASET_FINGERPRINT
assert checks['test_split_evaluated'] is False
rows = [
    {
        'training_id': row['target_id'],
        'training_name': row['target_name'],
        'stable_registry_id': row['registry_id'],
        'source_name': row['source_name'],
    }
    for row in mapping['source_to_target']
]
print('Fingerprint:', checks['dataset_fingerprint'])
print('Images:', sum(checks['splits'][split]['images'] for split in ('train', 'val', 'test')))
rows

## Preflight

Preflight kiểm tra CUDA, Ultralytics, dataset fingerprint, class mapping, optimizer/augmentation và các output path. Nó không tạo model và không bắt đầu training.

In [ ]:
def kd_command(command: str, *extra: str) -> list[str]:
    command_line = [
        sys.executable, 'scripts/train_detector_kd.py', command,
        '--config', KD_CONFIG,
    ]
    if BASE_CONFIG_OVERRIDE:
        command_line.extend(['--base-config', BASE_CONFIG_OVERRIDE])
    command_line.extend(extra)
    return command_line

def run_guarded(command: list[str], enabled: bool, label: str) -> None:
    print(f'{label}:', subprocess.list2cmdline(command))
    if enabled:
        subprocess.run(command, cwd=PROJECT_ROOT, check=True)
    else:
        print(f'Không chạy {label}: flag=False')

## Xem trước augmentation thật trước khi train

Code hiện dùng augmentation online từ base config: HSV nhẹ, translate 0.05, scale 0.2, mosaic 0.3, không flip/mixup/copy-paste. Cell này chạy đúng Ultralytics train transform và vẽ lại bbox; nó không train model và không tạo thêm bản sao offline. Hãy mở `data/brand_21brands_yolo_v2/augmentation_previews/contact_sheet.jpg` để kiểm tra nhãn còn bám đúng sản phẩm.

In [ ]:
augmentation_config = BASE_CONFIG_OVERRIDE or 'configs/brand_21brands_yolo.yaml'
augmentation_command = [
    sys.executable, 'scripts/preview_yolo_augmentations.py',
    '--config', augmentation_config, '--count', '8', '--seed', '42',
]
run_guarded(augmentation_command, RUN_AUGMENTATION_PREVIEW, 'augmentation preview')

## Pilot hyperparameter tuning trước full training

Không thể biết parameter tốt nhất nếu chưa đo validation. Search nhỏ này chạy 5 trial × 25 epoch, chỉ thay learning rate và mức mosaic/scale; test split không được dùng. Kết quả tạo `runs/model1_brand21_tuning/recommended_full_config.yaml`. Đây là ứng viên tốt nhất trong grid hữu hạn, không phải tối ưu toàn cục. Sau khi tuning hoàn tất, đặt `BASE_CONFIG_OVERRIDE` tới file đó rồi chạy lại KD preflight.

In [ ]:
tuning_base_config = 'configs/brand_21brands_yolo.yaml'
tuning_base_command = [
    sys.executable, 'scripts/tune_detector_hyperparameters.py',
    '--config', tuning_base_config,
    '--search', HYPERPARAMETER_SEARCH_CONFIG,
]
run_guarded(
    [*tuning_base_command, '--preflight'],
    RUN_TUNING_PREFLIGHT,
    'hyperparameter tuning preflight',
)
run_guarded(
    [*tuning_base_command, '--run'],
    RUN_HYPERPARAMETER_TUNING,
    'hyperparameter pilot search',
)

In [ ]:
run_guarded(kd_command('preflight'), RUN_PREFLIGHT, 'preflight')

## Bước 1 — train hoặc resume teacher

Teacher mặc định là YOLOv8s. Khi run chưa tồn tại, bật `RUN_TEACHER`. Nếu bị gián đoạn, tắt cờ đó và bật `RUN_RESUME_TEACHER`; resume phải dùng chính `weights/last.pt`, không dùng `best.pt`.

In [ ]:
run_guarded(kd_command('teacher'), RUN_TEACHER, 'train teacher')
run_guarded(
    kd_command('resume-teacher', '--resume', TEACHER_LAST),
    RUN_RESUME_TEACHER,
    'resume teacher',
)

## Bước 2 — train hoặc chọn baseline YOLOv8n

Baseline là đối chứng bắt buộc để biết KD có thực sự tốt hơn hay không. Có thể train baseline mới bằng cell này. Nếu đã có baseline 21-brand v2 hoàn chỉnh, giữ cờ `False` và đổi `BASELINE_CHECKPOINT` tới `best.pt` đó.

In [ ]:
run_guarded(kd_command('baseline'), RUN_BASELINE, 'train baseline')
run_guarded(
    kd_command('resume-baseline', '--resume', BASELINE_LAST),
    RUN_RESUME_BASELINE,
    'resume baseline',
)

## Bước 3 — train hoặc resume KD student

KD giữ nguyên supervised YOLO losses rồi cộng class-logit và DFL distillation loss. Teacher phải hoàn thành và có cùng 21 class theo đúng thứ tự. Batch KD mặc định là 2 để giảm nguy cơ thiếu VRAM; có thể chỉnh trong YAML dựa trên GPU server.

In [ ]:
run_guarded(
    kd_command('student-kd', '--teacher', TEACHER_CHECKPOINT),
    RUN_STUDENT_KD,
    'train KD student',
)
run_guarded(
    kd_command(
        'resume-kd', '--teacher', TEACHER_CHECKPOINT,
        '--resume', KD_STUDENT_LAST,
    ),
    RUN_RESUME_KD,
    'resume KD student',
)

## Bước 4 — so sánh trên validation

Chỉ bật sau khi có đủ ba checkpoint. Báo cáo so sánh precision, recall, mAP50 và mAP50–95 trên validation; test split không được đọc. Checkpoint deploy của KD là `best_portable.pt`, không phải checkpoint custom dùng để resume.

In [ ]:
compare_command = kd_command(
    'compare',
    '--teacher', TEACHER_CHECKPOINT,
    '--baseline', BASELINE_CHECKPOINT,
    '--student', KD_STUDENT_CHECKPOINT,
)
run_guarded(compare_command, RUN_COMPARE, 'compare validation metrics')

## Bước 5 — hiệu chỉnh inference sau khi đã chốt checkpoint

`conf`, NMS `iou` và `imgsz` tốt nhất phụ thuộc checkpoint và mục tiêu vận hành nên không thể chọn trước training. Grid này chạy 12 lần trên validation, mặc định chọn F1 cao nhất và ghi cả latency. Không dùng test. Chỉ bật sau khi có `best_portable.pt`; kết quả nằm trong `runs/model1_brand21_inference_calibration/inference_calibration.json`.

In [ ]:
inference_base_config = BASE_CONFIG_OVERRIDE or 'configs/brand_21brands_yolo.yaml'
inference_base_command = [
    sys.executable, 'scripts/calibrate_detector_inference.py',
    '--config', inference_base_config,
    '--search', INFERENCE_SEARCH_CONFIG,
    '--weights', KD_STUDENT_CHECKPOINT,
    '--output-dir', 'runs/model1_brand21_inference_calibration',
]
run_guarded(
    inference_base_command,
    RUN_INFERENCE_PREFLIGHT,
    'inference calibration preflight',
)
run_guarded(
    [*inference_base_command, '--run'],
    RUN_INFERENCE_CALIBRATION,
    'inference calibration',
)

## Theo dõi artifact

`results.csv` chứa metric Ultralytics; KD student có thêm `kd_metrics.csv`. `last.pt` dùng resume, còn `best_portable.pt` dùng validation/deploy sau khi KD hoàn tất.

In [ ]:
artifact_paths = [
    Path(TEACHER_CHECKPOINT), Path(BASELINE_CHECKPOINT),
    Path(KD_STUDENT_CHECKPOINT), Path(KD_STUDENT_LAST),
    Path('runs/model1_brand21_distillation/student_yolov8n_kd_100ep_s42/results.csv'),
    Path('runs/model1_brand21_distillation/student_yolov8n_kd_100ep_s42/kd_metrics.csv'),
    Path('runs/model1_brand21_distillation/comparison_report.json'),
    Path('runs/model1_brand21_tuning/tuning_summary.json'),
    Path('runs/model1_brand21_tuning/recommended_full_config.yaml'),
    Path('runs/model1_brand21_inference_calibration/inference_calibration.json'),
]
[
    {
        'path': path.as_posix(),
        'exists': path.is_file(),
        'size_bytes': path.stat().st_size if path.is_file() else 0,
    }
    for path in artifact_paths
]

## Lưu ý vận hành

Đóng tab trình duyệt thường không dừng kernel Jupyter trên server, nhưng restart/terminate kernel sẽ dừng subprocess đang train. Luôn kiểm tra `last.pt` trước khi shutdown. Không resume checkpoint 9-brand hoặc checkpoint v1 registry-order vào workflow này.